# Build Global City Points Layer (`AllCities_Points.gpkg`)

This notebook aggregates per-country RF prediction GeoPackages into a single
city-level point layer used as a figure support file for Figure 2 and the Global
Summary Table.

**Input:** Per-country RF prediction GeoPackages in `data_external/zenodo/predictions/`  
These are large files, not committed to GitHub. Download from Zenodo  
([DOI: 10.5281/zenodo.20486977](https://doi.org/10.5281/zenodo.20486977)).

**Output:** `4_Figures_Tables/AllCities_Points.gpkg`  
This is a small derived file retained in the repository and tracked in GitHub.
It is used directly by `01_Figure2_Global_DeprivedShare.ipynb` and
`05_GlobalSummaryTable.ipynb` without requiring re-computation from the raw
prediction GeoPackages.

In [ ]:
# ============================================================
# Build global city points with population & area aggregations
# (EXCLUDING selected countries)
# ============================================================
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np

# ============================================================
# PATH CONFIGURATION  (portable — no hard-coded local paths)
# ============================================================
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "citysegmentdeprivation" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATA_EXTERNAL      = REPO_ROOT / "data_external"
ZENODO_DATA        = DATA_EXTERNAL / "zenodo"
PREDICTIONS_DIR    = ZENODO_DATA / "predictions"
FIGURES_TABLES_DIR = REPO_ROOT / "4_Figures_Tables"

# --------- CONFIG ---------
IN_DIR   = PREDICTIONS_DIR
PATTERN  = "*_rf_preds.gpkg"
OUT_DIR  = FIGURES_TABLES_DIR
OUT_GPKG = OUT_DIR / "AllCities_Points.gpkg"

# Countries to exclude (based on filename stem, lowercase) - these countries are outside Africa, Asia, LAC
EXCLUDE_COUNTRIES = {
    "moldova",
    "fiji",
    "papua_new_guinea",
    "solomon_islands",
}

# Column names (expected in input GPKGs)
COL_CITY   = "UC_NM_MN"
COL_COUNTRY= "CTR_MN_NM"
COL_REGION = "REG1_GHSL"
COL_POP    = "POP_SEG"
COL_LABEL  = "rf_label"

CRS_AREA   = "EPSG:6933"  # World Cylindrical Equal Area (meters)
CRS_OUT    = "EPSG:4326"  # final output

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("IN_DIR:  ", IN_DIR)
print("OUT_GPKG:", OUT_GPKG)

In [ ]:
def _check_columns(gdf: gpd.GeoDataFrame, required):
    missing = [c for c in required if c not in gdf.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}")

def _aggregate_city_points(gdf_in: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Takes a country's segment GeoDataFrame in *any CRS*, computes area in EPSG:6933,
    aggregates per city, and returns city centroids (EPSG:4326) with metrics.
    """
    # Drop invalid/empty geometries early
    gdf_in = gdf_in[gdf_in.geometry.notna() & ~gdf_in.geometry.is_empty].copy()
    if gdf_in.empty:
        return gpd.GeoDataFrame(
            columns=[
                "UC_NM_MN", "CTR_MN_NM", "REG1_GHSL",
                "POP_TOTAL", "DEPRIVED_SEGS", "DEPRIVED_POP",
                "AREA_TOTAL", "AREA_DEPRIVED", "geometry"
            ],
            geometry="geometry",
            crs=CRS_OUT
        )

    # Ensure required columns
    _check_columns(gdf_in, [COL_CITY, COL_COUNTRY, COL_REGION, COL_POP, COL_LABEL])

    # Reproject to equal-area CRS for area + centroid math
    g_area = gdf_in.to_crs(CRS_AREA)

    # Compute per-segment area (m^2)
    g_area["AREA_SEG"] = g_area.geometry.area

    # Prepare flags
    g_area["_is_depr"] = (g_area[COL_LABEL] == 1)

    # ---- Aggregations per city ----
    agg_num = g_area.groupby(COL_CITY).agg(
        POP_TOTAL     = (COL_POP, "sum"),
        DEPRIVED_SEGS = ("_is_depr", "sum"),
        DEPRIVED_POP  = (COL_POP, lambda s: s[g_area.loc[s.index, "_is_depr"]].sum()),
        AREA_TOTAL    = ("AREA_SEG", "sum"),
        AREA_DEPRIVED = ("AREA_SEG", lambda s: s[g_area.loc[s.index, "_is_depr"]].sum()),
    )

    agg_cat = g_area.groupby(COL_CITY).agg(
        CTR_MN_NM = (COL_COUNTRY, "first"),
        REG1_GHSL = (COL_REGION, "first"),
    )

    # Geometry: dissolve city footprint then centroid (in area CRS), then to EPSG:4326
    city_geoms = (
        g_area[[COL_CITY, "geometry"]]
        .dissolve(by=COL_CITY, as_index=True)  # MultiPolygon per city
        .centroid                              # centroid in EPSG:6933
        .to_crs(CRS_OUT)                       # output CRS
    )
    city_geoms.name = "geometry"

    # Combine
    df_city = pd.concat([agg_cat, agg_num, city_geoms], axis=1).reset_index()
    gdf_city = gpd.GeoDataFrame(df_city, geometry="geometry", crs=CRS_OUT)

    # Clean numeric types
    for col in ["POP_TOTAL", "DEPRIVED_SEGS", "DEPRIVED_POP", "AREA_TOTAL", "AREA_DEPRIVED"]:
        if col in gdf_city.columns:
            gdf_city[col] = pd.to_numeric(gdf_city[col], errors="coerce").astype(float)

    # Final schema (explicit order)
    gdf_city = gdf_city.rename(columns={COL_CITY: "UC_NM_MN"})[
        ["UC_NM_MN", "CTR_MN_NM", "REG1_GHSL",
         "POP_TOTAL", "DEPRIVED_SEGS", "DEPRIVED_POP",
         "AREA_TOTAL", "AREA_DEPRIVED", "geometry"]
    ]

    return gdf_city

In [ ]:
# --------- MAIN LOOP ---------
all_points = []

all_paths = sorted(IN_DIR.glob(PATTERN))
gpkg_paths = [
    p for p in all_paths
    if not any(country in p.stem.lower() for country in EXCLUDE_COUNTRIES)
]

print(f"📦 Found {len(all_paths)} files total")
print(f"🚫 Excluding {len(all_paths) - len(gpkg_paths)} files: {sorted(EXCLUDE_COUNTRIES)}")
print(f"📦 Processing {len(gpkg_paths)} files")

for i, p in enumerate(gpkg_paths, 1):
    try:
        gdf = gpd.read_file(p)

        # If input has no CRS, we do NOT guess — stop for this file
        if gdf.crs is None:
            raise ValueError(f"{p.name} has no CRS — please set it before running.")

        # Ensure numeric POP + rf_label
        for col in [COL_POP, COL_LABEL]:
            if col in gdf.columns:
                gdf[col] = pd.to_numeric(gdf[col], errors="coerce")

        # Drop rows missing required values
        gdf = gdf.dropna(subset=[COL_CITY, COL_POP, COL_LABEL])

        g_city = _aggregate_city_points(gdf)
        if not g_city.empty:
            g_city["__source_file"] = p.name  # optional breadcrumb
            all_points.append(g_city)

        print(f"  • {i}/{len(gpkg_paths)} {p.name}: {len(g_city)} city points")

    except Exception as e:
        print(f"⚠️ Skipping {p.name}: {e}")

# --------- CONCAT & SAVE ---------
if all_points:
    g_all = gpd.GeoDataFrame(
        pd.concat(all_points, ignore_index=True),
        geometry="geometry",
        crs=CRS_OUT
    )

    # Optional de-dup (in case same city appears in multiple files)
    g_all = g_all.drop_duplicates(subset=["UC_NM_MN", "CTR_MN_NM", "REG1_GHSL"]).reset_index(drop=True)

    g_all.to_file(OUT_GPKG, driver="GPKG")
    print("\n✅ Saved:", OUT_GPKG)
    print(f"Total cities: {len(g_all)}")
    print("Sample columns:", list(g_all.columns))
else:
    print("⚠️ No city points produced.")